# ECE 539 — P4 Integration with P1 Ridge Predictions
**Team G16 | Seungjae Chung**

This version uses **actual P1 Ridge model predictions** on DEAM's 75-song **development** split. **P3 CLAP scores are still simulated** to test the ranking interface. Therefore, the top-10 recommendations are **not validated real music recommendations**. P2 MERT predictions are not integrated yet.



## 1. Install compatible scikit-learn and configure the prototype
P1's serialized pipeline was created with scikit-learn 1.6.1. Loading it in newer versions can fail. Only load `.joblib` files obtained from trusted teammates, since joblib uses pickle.

In [1]:
%pip -q install scikit-learn==1.6.1
import io
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

if sklearn.__version__ != "1.6.1":
    raise RuntimeError(f"scikit-learn {sklearn.__version__} loaded. Restart the Colab runtime, then Run all again.")
SEED = 539
TOP_K = 10
EMOTION_WEIGHT = 0.5
EVALUATION_SPLIT = "dev"
DATA_MODE = "P1 Ridge real predictions + MOCK CLAP"
MOODS = {
    "Relax":   {"valence": 6.5, "arousal": 2.5, "prompt": "calm, relaxing music"},
    "Study":   {"valence": 5.5, "arousal": 3.0, "prompt": "gentle, unobtrusive music for studying"},
    "Workout": {"valence": 7.0, "arousal": 8.0, "prompt": "energetic music for exercising"},
    "Happy":   {"valence": 8.0, "arousal": 6.5, "prompt": "happy, upbeat music"},
    "Sad":     {"valence": 2.5, "arousal": 3.0, "prompt": "sad, reflective music"},
}
print(DATA_MODE)


P1 Ridge real predictions + MOCK CLAP


## 2. Load P1 files and generate actual Ridge predictions
Upload these four files: `deam_combined_500.csv`, `song_split_ids(1).csv` (or `song_split_ids.csv`), `ridge_alpha100.joblib`, and `feature_columns.joblib`.

The notebook predicts on **dev only** (75 songs). Ground-truth `valence_mean`/`arousal_mean` are **not used for ranking**. The Ridge model is P1's baseline, **not** P2's MERT model.

The `clap_*` columns are **simulated placeholders**. Their scores are *not* produced by CLAP and cannot be used to assess recommendation quality.


In [2]:
from google.colab import files
print("Upload the four P1 files together:")
uploaded = files.upload()

def pick_file(name):
    matches = [k for k in uploaded if k == name]
    if name == "song_split_ids.csv" and not matches:
        matches = [k for k in uploaded if k.startswith("song_split_ids") and k.endswith(".csv")]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one {name}, found: {matches}")
    return io.BytesIO(uploaded[matches[0]])

combined = pd.read_csv(pick_file("deam_combined_500.csv"))
split_ids = pd.read_csv(pick_file("song_split_ids.csv"))
model = joblib.load(pick_file("ridge_alpha100.joblib"))
feature_columns = joblib.load(pick_file("feature_columns.joblib"))
assert len(feature_columns) == 260
assert combined["song_id"].is_unique and split_ids["song_id"].is_unique
assert set(combined["song_id"]) == set(split_ids["song_id"])
assert set(feature_columns).issubset(combined.columns)

dev = combined.merge(split_ids, on="song_id", validate="one_to_one")
dev = dev.loc[dev["split"] == "dev"].copy()
assert len(dev) == 75, f"Expected 75 dev songs, got {len(dev)}"
predictions = model.predict(dev[feature_columns])
assert predictions.shape == (len(dev), 2), predictions.shape
songs = pd.DataFrame({
    "song_id": dev["song_id"].to_numpy(),
    "valence": predictions[:, 0],
    "arousal": predictions[:, 1],
    "split": "dev",
})
# P1 model outputs may extend beyond label limits; clip for prototype similarity.
for col in ["valence", "arousal"]:
    n_clipped = (~songs[col].between(1, 9)).sum()
    if n_clipped:
        print(f"Clipping {n_clipped} {col} predictions to 1–9 for ranking")
        songs[col] = songs[col].clip(1, 9)

# Placeholder scores ONLY: replace with real P3 CLAP scores, joined by song_id.
rng = np.random.default_rng(SEED)
for mood, cfg in MOODS.items():
    dist = np.sqrt(((songs["valence"] - cfg["valence"]) / 8)**2 +
                   ((songs["arousal"] - cfg["arousal"]) / 8)**2) / np.sqrt(2)
    songs[f"clap_{mood}"] = np.clip(0.85 - dist + rng.normal(0, 0.12, len(songs)), 0, 1)
print(f"Loaded {len(songs)} actual P1 dev predictions; CLAP columns are MOCK")
display(songs.head())


Upload the four P1 files together:


Saving deam_combined_500.csv to deam_combined_500.csv
Saving feature_columns.joblib to feature_columns.joblib
Saving ridge_alpha100.joblib to ridge_alpha100.joblib
Saving song_split_ids.csv to song_split_ids.csv
Loaded 75 actual P1 dev predictions; CLAP columns are MOCK


,song_id,valence,arousal,split,clap_Relax,clap_Study,clap_Workout,clap_Happy,clap_Sad
0,286,3.924397,4.628716,dev,0.573548,0.723574,0.549464,0.460265,0.788326
1,1138,4.371282,3.923897,dev,0.490257,0.897639,0.415722,0.509961,0.585786
2,1946,5.485996,5.797673,dev,0.413695,0.516969,0.467566,0.605998,0.460014
3,120,3.907858,4.208008,dev,0.607495,0.808027,0.359891,0.418444,0.865982
4,1154,4.666729,3.846731,dev,0.757071,0.687311,0.494806,0.626156,0.674185


## 3. Validate data and calculate scores

**Emotion match:** normalized Euclidean distance in 2D valence/arousal space (1–9). Similarity = `1 − distance`.

**CLAP match:** per-mood raw CLAP scores are min–max normalized across the candidate pool. This is a prototype ranking choice, **not** calibrated probability. Constant scores get a neutral 0.5.

**Combined score:** `emotion_weight × emotion_similarity + (1 − emotion_weight) × clap_similarity`. Rank descending, break ties by `song_id`. Restrict to the development split if split labels are supplied. We do not claim this weighting is optimal.

In [3]:
def validate_songs(df):
    required = ["song_id", "valence", "arousal"] + [f"clap_{m}" for m in MOODS]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    if df["song_id"].isna().any() or df["song_id"].duplicated().any():
        raise ValueError("song_id must be nonmissing and unique; check joins and split files")
    for col in required[1:]:
        values = pd.to_numeric(df[col], errors="coerce")
        if not np.isfinite(values.to_numpy(dtype=float)).all():
            raise ValueError(f"{col} contains missing or nonnumeric values")
    if ((df[["valence", "arousal"]].astype(float) < 1) |
        (df[["valence", "arousal"]].astype(float) > 9)).any().any():
        raise ValueError("Valence/arousal must be on the 1–9 scale. Convert model outputs first.")
    if "split" in df.columns:
        labels = df["split"].astype(str).str.lower()
        if not labels.isin(["train", "dev", "test"]).all():
            raise ValueError("split must contain only train, dev or test")
    return True

def minmax_normalize(values):
    x = np.asarray(values, dtype=float)
    lo, hi = x.min(), x.max()
    return np.full_like(x, 0.5) if np.isclose(lo, hi) else (x - lo) / (hi - lo)

def recommend(df, mood="Relax", k=TOP_K, emotion_weight=EMOTION_WEIGHT, split=EVALUATION_SPLIT):
    validate_songs(df)
    if mood not in MOODS:
        raise ValueError(f"Unknown mood: {mood}")
    if not 0 <= emotion_weight <= 1:
        raise ValueError("emotion_weight must be between 0 and 1")
    if k < 1:
        raise ValueError("k must be positive")
    candidates = df.copy()
    if "split" in candidates.columns:
        candidates = candidates[candidates["split"].astype(str).str.lower() == split].copy()
    if len(candidates) < k:
        raise ValueError(f"Need at least {k} songs in split '{split}', found {len(candidates)}")
    cfg = MOODS[mood]
    distance = np.sqrt(((candidates["valence"].astype(float) - cfg["valence"]) / 8)**2 +
                       ((candidates["arousal"].astype(float) - cfg["arousal"]) / 8)**2) / np.sqrt(2)
    candidates["emotion_similarity"] = 1 - distance
    candidates["clap_similarity"] = minmax_normalize(candidates[f"clap_{mood}"].astype(float))
    candidates["combined_score"] = (emotion_weight * candidates["emotion_similarity"] +
                                    (1 - emotion_weight) * candidates["clap_similarity"])
    candidates["mood"] = mood
    candidates = candidates.sort_values(["combined_score", "song_id"], ascending=[False, True])
    cols = ["song_id", "mood", "valence", "arousal", f"clap_{mood}",
            "emotion_similarity", "clap_similarity", "combined_score"]
    return candidates[cols].head(k).reset_index(drop=True)

validate_songs(songs)
print("Validation passed.")
display(recommend(songs, "Relax"))

Validation passed.


,song_id,mood,valence,arousal,clap_Relax,emotion_similarity,clap_similarity,combined_score
0,1216,Relax,5.751825,5.109248,0.848225,0.760079,0.974368,0.867224
1,501,Relax,3.986977,4.575258,0.861252,0.711930,0.994732,0.853331
2,1902,Relax,5.659231,5.741480,0.864623,0.704010,1.000000,0.852005
3,399,Relax,5.771197,5.535542,0.809117,0.724069,0.913239,0.818654
4,1154,Relax,4.666729,3.846731,0.757071,0.798937,0.831886,0.815411
5,168,Relax,3.702539,3.798157,0.795180,0.727411,0.891455,0.809433
6,1131,Relax,4.261988,3.928544,0.767336,0.765322,0.847931,0.806626
7,1691,Relax,4.100610,4.156155,0.775277,0.742307,0.860343,0.801325
8,1716,Relax,6.328160,4.950239,0.737010,0.782895,0.800528,0.791712
9,1283,Relax,5.791311,4.540088,0.719120,0.809110,0.772565,0.790838


## 4. Interactive dropdown
Choose a mood and adjust the emotion weight. **Emotion values are P1 predictions; CLAP values are synthetic.** Results are only for integration testing.

In [4]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    dropdown = widgets.Dropdown(options=list(MOODS), value="Relax", description="Mood:")
    slider = widgets.FloatSlider(value=EMOTION_WEIGHT, min=0, max=1, step=0.1,
                                 description="Emotion weight:", style={"description_width": "initial"})
    out = widgets.Output()
    def show_results(change=None):
        with out:
            clear_output(wait=True)
            print("Mode:", DATA_MODE, "| Prompt:", MOODS[dropdown.value]["prompt"])
            display(recommend(songs, dropdown.value, emotion_weight=slider.value))
    dropdown.observe(show_results, names="value")
    slider.observe(show_results, names="value")
    display(widgets.VBox([dropdown, slider, out]))
    show_results()
except ImportError:
    print("ipywidgets unavailable; choose a mood manually:")
    display(recommend(songs, "Study"))

## 5. Automated software tests
Check top-10 uniqueness, score order, input validation, and mood responsiveness. Passing these tests **does not establish recommendation accuracy**.

In [5]:
for mood in MOODS:
    result = recommend(songs, mood)
    assert len(result) == TOP_K
    assert result["song_id"].is_unique
    assert result["combined_score"].is_monotonic_decreasing
    assert result["combined_score"].between(0, 1).all()

assert recommend(songs, "Relax")["song_id"].tolist() != recommend(songs, "Workout")["song_id"].tolist(), "Rankings should respond to mood"
try:
    recommend(songs, "InvalidMood")
    raise AssertionError("Invalid mood was not rejected")
except ValueError:
    pass
try:
    validate_songs(songs.drop(columns=["clap_Relax"]))
    raise AssertionError("Missing CLAP column was not rejected")
except ValueError:
    pass
try:
    validate_songs(pd.concat([songs, songs.iloc[[0]]], ignore_index=True))
    raise AssertionError("Duplicate song ID was not rejected")
except ValueError:
    pass
print(f"PASS: all {len(MOODS)} moods return {TOP_K} unique ranked songs; error checks pass.")

PASS: all 5 moods return 10 unique ranked songs; error checks pass.


## 6. Export integration-test results
These outputs contain **real P1 Ridge emotion predictions and synthetic CLAP scores**. They are not final recommendations.

In [6]:
all_results = pd.concat([recommend(songs, mood) for mood in MOODS], ignore_index=True)
filename = "P4_P1_RIDGE_MOCK_CLAP_DEV_top10.csv"
columns = ["song_id", "mood", "valence", "arousal", "emotion_similarity", "clap_similarity", "combined_score"]
all_results[columns].to_csv(filename, index=False)
print(f"Saved {filename} with {len(all_results)} rows")
display(all_results[columns].head())
# To download: files.download(filename)


Saved P4_P1_RIDGE_MOCK_CLAP_DEV_top10.csv with 50 rows


,song_id,mood,valence,arousal,emotion_similarity,clap_similarity,combined_score
0,1216,Relax,5.751825,5.109248,0.760079,0.974368,0.867224
1,501,Relax,3.986977,4.575258,0.711930,0.994732,0.853331
2,1902,Relax,5.659231,5.741480,0.704010,1.000000,0.852005
3,399,Relax,5.771197,5.535542,0.724069,0.913239,0.818654
4,1154,Relax,4.666729,3.846731,0.798937,0.831886,0.815411


## 7. Status / next steps

- [x] P1 processed DEAM features and 500-song split loaded
- [x] P1 trained Ridge pipeline used to predict valence/arousal for 75 dev songs
- [x] P4 mood-based ranking, dropdown, tests and CSV export
- [ ] Replace synthetic `clap_*` columns with **real P3 CLAP prompt-specific scores** joined on `song_id`
- [ ] Replace P1 baseline emotion predictions with **P2 MERT/MLP outputs** joined on `song_id`
- [ ] Validate actual song metadata/audio playback and evaluate mood-hit@10
- [ ] Keep held-out **test** songs unused until final evaluation
